# SWARM 2-Slice: Add Monitoring to a LIVE Deployment

Modifies the already-running `SWARM-MULTI-<N>-p1/p2` slices **in place** (no re-provisioning):

1. Adds a second `NIC_Basic` (`nic2`) to every existing agent + the database.
2. Creates a per-site FABNetv4 monitoring network (`fabv4mon-<site>`) in each slice and attaches every `nic2` to it.
3. Adds a new `monitor` VM (slice 1) whose single NIC joins its site's monitoring network; it runs Prometheus + Grafana (from `Prometheus_Grafana_Monitor/`).
4. Configures IPs/routes on the new NICs only (nic1 config is untouched), appends `<host>-mon` entries to `/etc/hosts`, installs `node_exporter` everywhere, and provisions the monitor.

**Prerequisites:** both slice parts are `StableOK` with SSH up, and the original
`SWARM-2slice.ipynb` setup (node_tools upload, netplan on nic1, /etc/hosts) has already run.

**Run the modify cells ONCE per slice** -- re-running them would try to add a duplicate `nic2` component.


## Import the libraries

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager(project_id="3a05ccb3-a4b9-4bc8-9bc8-4c8eb65c9d3e", token_location="./id_token.json")

fablib.show_config();


## Define variables (must match the running deployment)

In [ ]:
name_prefix = "agent"
total_agents = 100
base_slice_name = "SWARM-MULTI"

slice_name_1 = f"{base_slice_name}-{total_agents}-p1"
slice_name_2 = f"{base_slice_name}-{total_agents}-p2"

db_node_name = "database"
image = "docker_ubuntu_22"

# ============================================================
# MONITORING
# Every existing VM gets a second NIC (nic2) on a per-site FABNetv4
# monitoring network. A new "monitor" VM (slice 1) runs
# Prometheus + Grafana and scrapes node_exporter on every node.
# ============================================================
monitor_node_name = "monitor"
monitor_cores = 4
monitor_ram = 16
monitor_disk = 100           # disk is the binding placement constraint; keep small
monitor_site = "RUTG"        # must be a site that already has nodes in slice 1
mon_network_name = "fabv4mon"   # per-site monitoring networks (nic2)


## Modify slice part 1 (run ONCE)

Adds `nic2` to the database and every agent in part 1, one `fabv4mon-<site>`
network per site, and the new `monitor` VM on `monitor_site`.


In [ ]:
slice1 = fablib.get_slice(slice_name_1)

# Group the existing nodes by site (derived live -- no dependence on the plan dicts)
nodes1 = list(slice1.get_nodes())
sites1 = {}
for n in nodes1:
    sites1.setdefault(n.get_site(), []).append(n)

assert monitor_site in sites1, f"monitor_site {monitor_site} has no nodes in slice 1"

for site, site_nodes in sites1.items():
    # New site-scoped FABNetv4 L3 network for monitoring traffic (nic2).
    # FABNetv4 is routable across slices (10.128.0.0/10), so the monitor
    # reaches every site's monitoring network in BOTH slices.
    mon_net = slice1.add_l3network(name=f"{mon_network_name}-{site}", type="IPv4")
    print(f"{site}: fabv4mon network + nic2 on {len(site_nodes)} existing nodes")
    for n in site_nodes:
        mon_iface = n.add_component(model="NIC_Basic", name="nic2").get_interfaces()[0]
        mon_iface.set_mode("manual")
        mon_net.add_interface(mon_iface)

    # New monitor VM (Prometheus + Grafana), single NIC on its site's mon network
    if site == monitor_site:
        monitor = slice1.add_node(name=monitor_node_name, site=monitor_site,
                                  image=image, disk=monitor_disk,
                                  cores=monitor_cores, ram=monitor_ram)
        mon_vm_iface = monitor.add_component(model="NIC_Basic", name="nic1").get_interfaces()[0]
        mon_vm_iface.set_mode("manual")
        mon_net.add_interface(mon_vm_iface)
        print(f"{site}: + monitor VM ({monitor_cores}c/{monitor_ram}g/{monitor_disk}d)")

# Submit the slice modification
slice1.submit(wait=False)


## Modify slice part 2 (run ONCE)

In [ ]:
slice2 = fablib.get_slice(slice_name_2)

nodes2 = list(slice2.get_nodes())
sites2 = {}
for n in nodes2:
    sites2.setdefault(n.get_site(), []).append(n)

for site, site_nodes in sites2.items():
    mon_net = slice2.add_l3network(name=f"{mon_network_name}-{site}", type="IPv4")
    print(f"{site}: fabv4mon network + nic2 on {len(site_nodes)} existing nodes")
    for n in site_nodes:
        mon_iface = n.add_component(model="NIC_Basic", name="nic2").get_interfaces()[0]
        mon_iface.set_mode("manual")
        mon_net.add_interface(mon_iface)

slice2.submit(wait=False)


## Wait for both modifications to land

In [ ]:
for sn in (slice_name_1, slice_name_2):
    s = fablib.get_slice(sn)
    s.wait()
    s.wait_ssh()
    s.post_boot_config()
    print(f"{sn}: modify complete")


In [ ]:
# Sanity check: the new networks and the monitor VM
for sn in (slice_name_1, slice_name_2):
    s = fablib.get_slice(sn)
    s.list_networks();


## Combine both slices and configure the new NICs

Only the `fabv4mon-*` networks are configured here -- nic1 config is untouched.
Routing: nic1 keeps the full `10.128.0.0/10` route; each node's `nic2` gets a
route ONLY to the monitor's monitoring subnet, so the two NICs never conflict.


In [ ]:
slice1 = fablib.get_slice(slice_name_1)
slice2 = fablib.get_slice(slice_name_2)

nodes = list(slice1.get_nodes()) + list(slice2.get_nodes())
node_by_name = {n.get_name(): n for n in nodes}

monitor_node = node_by_name.get(monitor_node_name)
assert monitor_node is not None, "monitor node not found -- did the slice 1 modify succeed?"

# Nodes that run SWARM (everything except the monitor VM)
swarm_nodes = [n for n in nodes if n.get_name() != monitor_node_name]

# Monitoring networks + their interfaces (expensive get_* calls, cache once)
networks = list(slice1.get_networks()) + list(slice2.get_networks())
mon_nw_by_name = {nw.get_name(): nw for nw in networks
                  if nw.get_name().startswith(mon_network_name)}
mon_nw_ifaces = {name: nw.get_interfaces() for name, nw in mon_nw_by_name.items()}

print(f"Combined fleet: {len(nodes)} nodes, {len(mon_nw_by_name)} monitoring networks")


In [ ]:
# Re-upload node_tools: setup-netplan-multihomed.sh gained per-interface
# netplan files + on-link route skipping, both required for the second NIC.
from concurrent.futures import ThreadPoolExecutor, as_completed

def push_tools(node):
    node.upload_directory("node_tools", ".")
    node.execute("cd node_tools && chmod +x *.sh", quiet=True)
    return node.get_name()

with ThreadPoolExecutor(max_workers=16) as pool:
    futures = [pool.submit(push_tools, n) for n in nodes]
    for f in as_completed(futures):
        print(f"node_tools updated on {f.result()}")


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# Assign IPs on the monitoring networks only
mon_assigned_ip = {}

# The monitor's monitoring subnet: every agent's nic2 gets a route to THIS
# subnet only. The full 10.128.0.0/10 route stays on nic1.
monitor_mon_nw = mon_nw_by_name[f"{mon_network_name}-{monitor_site}"]
monitor_mon_subnet = monitor_mon_nw.get_subnet()

node_cmds = {}   # node_name -> list of netplan commands

for nw_name, nw in mon_nw_by_name.items():
    subnet = nw.get_subnet()
    print(nw_name, subnet)
    hiter = subnet.hosts()
    ip = next(hiter)                     # Skip first host
    ip = next(hiter)                     # Skip first host
    for iface in mon_nw_ifaces[nw_name]:
        node_name = iface.get_node().get_name()

        if node_name == monitor_node_name:
            lan_net = "10.128.0.0/10"            # monitor reaches ALL monitoring subnets
        else:
            lan_net = str(monitor_mon_subnet)    # agents: only the monitor's subnet via nic2

        cmd = (
            f"sudo node_tools/setup-netplan-multihomed.sh "
            f"-i {iface.get_physical_os_interface_name()} "
            f"-a {ip}/24 "
            f"-n {lan_net} "
            f"-g {nw.get_gateway()}"
        )
        node_cmds.setdefault(node_name, []).append(cmd)
        mon_assigned_ip[(nw_name, node_name)] = str(ip)

        ip = next(hiter)

def config_node(node_name):
    node = node_by_name[node_name]
    for cmd in node_cmds[node_name]:
        node.execute(cmd, quiet=True, output_file=f"{node_name}-netplan-mon.log")
    return node_name

with ThreadPoolExecutor(max_workers=16) as pool:
    futures = [pool.submit(config_node, nn) for nn in node_cmds]
    for f in as_completed(futures):
        print(f"Configured {f.result()}")


In [ ]:
# Append "<host>-mon" entries (and the monitor itself) to /etc/hosts on every node
from concurrent.futures import ThreadPoolExecutor, as_completed

host_to_ip = {}
for (nw, host), ip in mon_assigned_ip.items():
    name = host if host == monitor_node_name else f"{host}-mon"
    host_to_ip.setdefault(name, ip)

block_lines = [f"{ip} {host}" for host, ip in sorted(host_to_ip.items())]
hosts_blocks = "\n".join(block_lines)

def append_hosts(node):
    # Idempotent-ish: drop any previous -mon/monitor lines before appending
    node.execute(
        "sudo sh -c \"sed -i '/-mon$/d; / monitor$/d' /etc/hosts && "
        f"echo '{hosts_blocks}' >> /etc/hosts\"", quiet=True)
    return node.get_name()

with ThreadPoolExecutor(max_workers=16) as pool:
    futures = [pool.submit(append_hosts, n) for n in nodes]
    for f in as_completed(futures):
        pass

print(hosts_blocks)


## Deploy Monitoring (Prometheus + Grafana)

Installs `node_exporter` on every SWARM node, then provisions the `monitor` VM
using the scripts and Grafana dashboards from `Prometheus_Grafana_Monitor/`.


In [ ]:
# Install node_exporter on every SWARM node (agents + database), in parallel
from concurrent.futures import ThreadPoolExecutor, as_completed

exporter_script = "Prometheus_Grafana_Monitor/tools/setup-exporter.sh"

def setup_exporter(node):
    node.upload_file(exporter_script, "setup-exporter.sh")
    node.execute("chmod +x setup-exporter.sh && sudo ./setup-exporter.sh",
                 quiet=True, output_file=f"{node.get_name()}-exporter.log")
    return node.get_name()

with ThreadPoolExecutor(max_workers=16) as pool:
    futures = [pool.submit(setup_exporter, n) for n in swarm_nodes]
    for f in as_completed(futures):
        print(f"node_exporter installed on {f.result()}")


In [ ]:
# Build prometheus.yml from the monitoring-NIC IPs and provision the monitor
import os

# Scrape targets: every SWARM node's nic2 IP on the fabv4mon-* networks
mon_targets = sorted(
    (host, ip) for (nw, host), ip in mon_assigned_ip.items()
    if host != monitor_node_name
)

lines = [
    "global:",
    "  scrape_interval: 15s",
    "  evaluation_interval: 15s",
    "",
    "scrape_configs:",
    "  - job_name: prometheus",
    "    static_configs:",
    "      - targets: ['localhost:9090']",
    "",
    "  - job_name: node",
    "    static_configs:",
    "      - targets:",
    "          - 'localhost:9100'",
    "        labels:",
    "          node: monitor",
]
for host, ip in mon_targets:
    lines += [
        "      - targets:",
        f"          - '{ip}:9100'",
        "        labels:",
        f"          node: {host}",
    ]
prometheus_yml = "\n".join(lines) + "\n"

os.makedirs("monitoring-config", exist_ok=True)
with open("monitoring-config/prometheus.yml", "w") as f:
    f.write(prometheus_yml)

mon_dir = "Prometheus_Grafana_Monitor"
monitor_node.execute(
    "mkdir -p monitoring-config/grafana-provisioning/datasources "
    "monitoring-config/grafana-provisioning/dashboards", quiet=True)
monitor_node.upload_file("monitoring-config/prometheus.yml",
                         "monitoring-config/prometheus.yml")
monitor_node.upload_file(f"{mon_dir}/build/grafana-datasource.yml",
                         "monitoring-config/grafana-provisioning/datasources/datasource.yml")
monitor_node.upload_file(f"{mon_dir}/build/grafana-dashboard-provider.yml",
                         "monitoring-config/grafana-provisioning/dashboards/dashboard.yml")
monitor_node.upload_file(f"{mon_dir}/build/resource-utilization.json",
                         "monitoring-config/grafana-provisioning/dashboards/resource-utilization.json")
monitor_node.upload_file(f"{mon_dir}/tools/setup-monitor.sh", "setup-monitor.sh")

# setup-monitor.sh reads ~/monitoring-config as root, so stage it there
monitor_node.execute("sudo rm -rf /root/monitoring-config && "
                     "sudo cp -r monitoring-config /root/monitoring-config", quiet=True)
stdout, stderr = monitor_node.execute("chmod +x setup-monitor.sh && sudo ./setup-monitor.sh",
                                      quiet=True, output_file="monitor-setup.log")
print(f"Monitor provisioned; Prometheus scraping {len(mon_targets) + 1} targets")


In [ ]:
# Verify scrape health and print access info
import time
time.sleep(30)   # give Prometheus a scrape cycle
stdout, stderr = monitor_node.execute(
    "curl -s localhost:9090/api/v1/targets | python3 -c \"import json,sys; "
    "ts=json.load(sys.stdin)['data']['activeTargets']; "
    "up=sum(1 for t in ts if t['health']=='up'); "
    "print(f'{up}/{len(ts)} targets up'); "
    "[print(' DOWN:', t['labels'].get('node', t['scrapeUrl'])) for t in ts if t['health']!='up']\"")

monitor_mon_ip = mon_assigned_ip[(f"{mon_network_name}-{monitor_site}", monitor_node_name)]
print(f"Monitor FABNetv4 IP: {monitor_mon_ip}")
print(f"Prometheus: http://{monitor_mon_ip}:9090  (from any node in either slice)")
print(f"Grafana:    http://{monitor_mon_ip}:3000  (anonymous viewer, no login)")
print()
print("From your laptop, tunnel through the FABRIC bastion:")
print(str(monitor_node.get_ssh_command()).replace(
    "ssh ", "ssh -L 3000:localhost:3000 -L 9090:localhost:9090 ", 1))


### Troubleshooting

- **`nic2` missing in the OS** (`ip link` shows no new interface): FABRIC hot-plugs the NIC, but if it doesn't appear, reboot the node (`sudo reboot`) and re-run `post_boot_config()` for that slice, then re-run the IP-configuration cell.
- **Duplicate-component error on re-running a modify cell**: the modification already landed; skip to the wait cell.
- **Targets DOWN in Prometheus**: check the agent's `nic2` route (`ip route | grep <monitor subnet>`) and that `node_exporter` is active (`systemctl status node_exporter`).
